For simplicity, we will generate the initial simple data with the following constrinats
- We will always order particle labels from least to most, i.e. pi.pj or ei.ej for j>i. 
- For ei.pj, the polarization will always be on the left regardless of the order
- We will generate a numerator of up to $r$ monomials over a single denominator which contains only momentum dot products, ensuring that the entire term has mass dimension $q$ and little group weight $\ell$. Recall that under a little group transformation $L^\mu_{~\nu}(k)k^\nu = k^\nu$, polarization vectors behave as $$L^\mu_{~\nu}(k)\epsilon_\lambda^\nu(k) = \sum_{\lambda\lambda'}D_{\lambda\lambda'}\epsilon_{\lambda'}^\nu(k) = e^{i\theta\lambda}\epsilon_{\lambda}^\nu(k)$$
-  ~~The range of $r$, $q$ and $\ell$ should be specified as parameters~~
- The number of gluons and gravitons should be specified, with gluon labels running $1\rightarrow n_{gluons}$ and gravitons $n_{gluons}+1 \rightarrow n_{gravitons}$

TODO:
- [x] There should be an equal number of $\epsilon_i$'s in each term of the numerator, but there aren't. Should pre-decide the number of gluons/scalars/gravitons etc.
- [x] Convert to sympy for easier manipulation

In [9]:
import random
import sympy as sp
import h5py
import numpy as np

# Some useful functions for later
class DP(sp.Function):
    @classmethod
    def eval(cls, a, b):
        # No automatic simplification
        return None

   # def _sympystr(self, printer):
        # For str() and ascii output
    #    a, b = self.args
        # Use the Unicode centered dot (U+22C5) for the dot product
    #    return f"{printer._print(a)} \u22C5 {printer._print(b)}"

    def _pretty(self, printer, **kwargs):
        # For pretty printing (used by sp.pprint).
        a_form = printer._print(self.args[0])
        b_form = printer._print(self.args[1])
        #dot_form = prettyForm("⋅")  # Unicode centered dot

        # Concatenate the pretty-printed components
        return a_form*b_form
    def _latex(self, printer):
        # For LaTeX output (via sp.latex() or in Jupyter notebooks).
        a, b = self.args
        return r"%s \cdot %s" % (printer._print(a), printer._print(b))
    
    def to_token_list(self):
        """
        Return a token list for this DP expression (or any Sympy expression).
        This method uses a recursive tree traversal.
        """
        return tokenize_expr(self)
    
def extract_scalar_factor(expr):
    """
    Given a sympy expression 'expr', return (c, remainder) so that
         expr == c * remainder
    where c is the numeric coefficient (including its sign) and
    remainder is the non-numeric part.
    """
    c, args = expr.as_coeff_mul()
    remainder = sp.Mul(*args) if args else sp.Integer(1)
    return (c, remainder)

def distribute_DP(expr):
    """
    Recursively distribute the DP dot product over additions in its arguments,
    and factor out any numeric coefficients so that DP(-p1, p2) becomes
    -DP(p1, p2).
    
    For example:
      DP(-p1, p2 + 2*p3)  ->  -DP(p1, p2) - 2*DP(p1, p3)
    """
    # Base case: if there's no DP or if the expression is atomic, return it unchanged.
    if not expr.has(DP) or expr.is_Atom:
        return expr

    # Handle the DP instance explicitly.
    if expr.func == DP:
        a, b = expr.args
        # Recursively expand the sub-arguments.
        a_expanded = distribute_DP(a)
        b_expanded = distribute_DP(b)
        
        # Factor out numeric coefficients from each argument.
        ca, a_noscale = extract_scalar_factor(a_expanded)
        cb, b_noscale = extract_scalar_factor(b_expanded)
        c_total = ca * cb
        dp_core = DP(a_noscale, b_noscale)
        
        # Distribute over sums if present in either argument.
        if a_noscale.is_Add and b_noscale.is_Add:
            terms = [distribute_DP(DP(term_a, term_b))
                     for term_a in a_noscale.args
                     for term_b in b_noscale.args]
            return c_total * sp.Add(*terms)
        elif a_noscale.is_Add:
            terms = [distribute_DP(DP(term, b_noscale))
                     for term in a_noscale.args]
            return c_total * sp.Add(*terms)
        elif b_noscale.is_Add:
            terms = [distribute_DP(DP(a_noscale, term))
                     for term in b_noscale.args]
            return c_total * sp.Add(*terms)
        else:
            # No sum to distribute over; simply return the DP multiplied by the scalar.
            return c_total * dp_core

    # Otherwise, apply the distribution recursively to the subexpressions.
    new_args = [distribute_DP(arg) for arg in expr.args]
    return expr.func(*new_args)

# Tokenization of a Sympy expression, for when we want to use it in our nn.
def tokenize_expr(expr):
    """
    Recursively tokenize a Sympy expression.
    
    The tokenization rules are:
      - For function applications, include the function name,
        a "(", the tokens of the arguments separated by a comma, and another ")".
      - For atomic expressions (symbols, numbers), just use str(expr).
      - For operators like Mul, Add, Pow, we replace them with shorter *,+,^.
    
    Returns a list of tokens.
    """
    tokens = []
    if expr.is_Atom:
        # For atoms (numbers, symbols), just use their string representation.
        tokens.append(str(expr))
    else:
        # For non-atomic expressions, include the function name.
        # (For example, an Add becomes "Add", a Mul becomes "Mul", etc.)
        func_name = expr.func.__name__
        if func_name == "Add":
            func_name = "+"
        elif func_name == "Mul":
            func_name = "*"
        elif func_name == "Pow":
            func_name = "^"
        tokens.append(func_name)
        tokens.append("(")
        # Process each argument.
        for i, arg in enumerate(expr.args):
            tokens.extend(tokenize_expr(arg))
            if i < len(expr.args) - 1:
                tokens.append(",")
        tokens.append(")")
    return tokens

# --- Helper: Check if a token represents a number ---
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

# --- The parser functions ---
def parse_tokens(tokens, pos=0):
    """
    Recursively parse a list of tokens (starting at position pos) into a Sympy expression.
    
    Grammar (roughly):
      expr   := atomic | function_call
      atomic := a token that is not followed by "("  (converted to a number or Symbol)
      function_call := token "(" arg_list ")"
      arg_list := expr ("," expr)*
      
    Additionally, for the operators 'Add', 'Mul', 'Pow', our tokenizer replaces them with '+', '*', '^'.
    In that case, we map them to sp.Add, sp.Mul, sp.Pow respectively.
    For the function "DP" we use our own DP function.
    For any other function name, we call sp.Function(name).
    """
    if pos >= len(tokens):
        raise ValueError("Ran out of tokens during parsing.")
        
    token = tokens[pos]
    
    # If the next token is "(", then we have a function call.
    if pos + 1 < len(tokens) and tokens[pos+1] == "(":
        func_token = token  # e.g., "DP", "+", "*", or "^"
        pos += 2  # Skip the function token and the "("
        args_list = []
        # Parse arguments until we hit a ")"
        while pos < len(tokens) and tokens[pos] != ")":
            arg, pos = parse_tokens(tokens, pos)
            args_list.append(arg)
            # If the next token is a comma, skip it.
            if pos < len(tokens) and tokens[pos] == ",":
                pos += 1
        if pos >= len(tokens) or tokens[pos] != ")":
            raise ValueError("Expected ')' at token position %d" % pos)
        pos += 1  # Skip the closing ")"
        
        # Map function token to actual function.
        if func_token == "+":
            expr = sp.Add(*args_list)
        elif func_token == "*":
            expr = sp.Mul(*args_list)
        elif func_token == "^":
            if len(args_list) != 2:
                raise ValueError("'^' operator expects exactly 2 arguments")
            expr = sp.Pow(args_list[0], args_list[1])
        elif func_token == "DP":
            expr = DP(*args_list)
        else:
            # For any other function name, create a generic Sympy function.
            f = sp.Function(func_token)
            expr = f(*args_list)
        return expr, pos
    else:
        # Atomic token: if it represents a number, convert it, otherwise a symbol.
        if is_number(token):
            # If it contains a dot, use Float; otherwise Integer.
            if '.' in token:
                expr = sp.Float(token)
            else:
                expr = sp.Integer(token)
        else:
            expr = sp.Symbol(token)
        return expr, pos + 1

def detokenize_expr(tokens):
    """
    Convert a token list back into a Sympy expression.
    
    Raises an error if not all tokens are consumed.
    """
    expr, pos = parse_tokens(tokens, 0)
    if pos != len(tokens):
        raise ValueError("Extra tokens remain after parsing: position %d of %d" % (pos, len(tokens)))
    return expr

# Let's test these functions    
expr = DP(sp.Symbol("a") - 3*sp.Symbol("b"),sp.Symbol("c") + 4*sp.Symbol("d"))
print(distribute_DP(expr))

# Get the token list using our built-in method.
tokens = tokenize_expr(DP(sp.Symbol("a"),sp.Symbol("b"))**2 + 3*DP(sp.Symbol("c"),sp.Symbol("d")))
print("Tokenized amplitude:")
sp.pprint(DP(sp.Symbol("a"),sp.Symbol("b"))**2 + 3*DP(sp.Symbol("c"),sp.Symbol("d")))
print(tokens)
expr_reconstructed = detokenize_expr(tokens)
print("\nReconstructed expression:")
sp.pprint(expr_reconstructed)

DP(a, c) + 4*DP(a, d) - 3*DP(b, c) - 12*DP(b, d)
Tokenized amplitude:
     2        
(a⋅b)  + 3⋅c⋅d
['+', '(', '^', '(', 'DP', '(', 'a', ',', 'b', ')', ',', '2', ')', ',', '*', '(', '3', ',', 'DP', '(', 'c', ',', 'd', ')', ')', ')']

Reconstructed expression:
     2        
(a⋅b)  + 3⋅c⋅d


In [10]:
def generate_monomial(n, n_gluons, n_gravitons, dim):
    """
    Generate a Sympy expression of total mass-dimension 'dim'.
    """
    
    # For gluons: exactly 1 copy of each label i in 1..n_gluons
    # For gravitons: 2 copies of each label j in n_gluons+1..n_gluons+n_gravitons. We assume that graviton polarization tensors are factorized into vectors.
    pol_indices = []
    for i in range(1, n_gluons + 1):
        pol_indices.append(i)
    for j in range(n_gluons + 1, n_gluons + n_gravitons + 1):
        pol_indices.append(j)
        pol_indices.append(j)
    
    total_pols = len(pol_indices)  # = n_gluons + 2*n_gravitons
    
    
    # We need to make sure the amplitude ends up having the correct mass dimension, noting that
    #   [pi.pj] = 2, [ei.pj] = 1, [ei.ej] = 0
    # The mass dimension (without coupling constants) is then given by
    #   dim = n(ei.pj) + n(pi.pj) = x + 2y
    # where x is the number of ei.pj factors and y is the number of pi.pj factors.
    # If we define z = n(ei.ej), then we have the constraints
    #   x + 2z = total_pols
    #   x + 2y = dim
    # This gives
    #   y = z + (dim - total_pols)/2
    # Recall that z is n(ei.ej), so it must be non-negative, an integer, and have a maximum total_pols//2.
    # 
    # This means we won't always have a valid solution to the above equation! In that case, we'll just return a trivial expression "1". Or maybe we should issue a warning?
    # Either way, we will try and solve this for z meeting the above constraints, and randomly choose a solution if there are more than 1.
    # We will also demand that x,y,z are all positive as we're trying to construct the amplitude numerator. The denominator is a seperate process.
    
    valid_solutions = []
    for z_candidate in range(total_pols//2 + 1):  # z can go up to total_pols//2
        x_candidate = total_pols - 2*z_candidate
        
        if x_candidate < 0:  # We want positive x
            continue
        # From y = z + (dim - total_pols)/2, we also need (dim - total_pols) to be even for y to be an integer
        shift = (dim - total_pols)
        if shift % 2 != 0:
            # We don't want fractional y values so we ignore any odd shifts!
            continue
        y_candidate = z_candidate + shift//2
        if y_candidate < 0: # We also want positive y
            continue
        # Checks passed - we have a valid triple (x_candidate, y_candidate, z_candidate)
        valid_solutions.append((x_candidate, y_candidate, z_candidate))
    
    # If no valid solutions, return a trivial expression "1". Maybe issue a warning here that no valid numerators?
    if not valid_solutions:
        return sp.Integer(1)
    
    # Choose a random solution for variety
    x, y, z = random.choice(valid_solutions)
    
    # Now we have: a set of gluon and graviton pol_indices, and how many of each type of factor we should include to satisfy the mass dimension constraints.
    # We now want to generate random factors that satisfy these constraints.
    # We'll:
    #   (a) Shuffle pol_indices, then pick x of them for e_i.p_j factors,
    #   (b) from the remainder, group them in pairs to form e_i.e_j factors,
    #   (c) build y p_i.p_j factors by picking random pairs of momentum indices,
    #   (d) shuffle them all, then form a product.

    factors = []
    
    # Shuffle and pick x polarizations for e_i.p_j
    random.shuffle(pol_indices)
    
    ep_indices = pol_indices[:x]   # e_i's that will go with p_j
    remaining  = pol_indices[x:]   # leftover for e_i.e_j
    # Annoyingly, after shuffling, these indices can have repeated indices side-by-side, meaning we could generate ei.ei terms (at least for gravitons). Need to figure out a way to avoid this.
    
    # build the e_i.p_j factors
    for e_idx in ep_indices:
        # choose a random momentum index in [1..n] that is not the same as e_idx
        momentum_choices = [mn for mn in range(1, n+1) if mn != e_idx]
        m_idx = random.choice(momentum_choices)
        e_sym = sp.Symbol(f"e{e_idx}", commutative=True)
        p_sym = sp.Symbol(f"p{m_idx}", commutative=True)
        factors.append(DP(e_sym, p_sym)) # polarizations to the left!
    
    # From the remaining polarization indices, pair them up for e_i.e_j
    # (We must have exactly 2z = len(remaining) from the valid solution.)

    for i in range(z):
        #print("Remaining:", remaining)
        # We need to remove any side-by-side duplicates to avoid ei.ei terms
        # Check for duplicates
        for _ in range(50):  # Try up to 50 times de-shuffle the remaining indices. In testing, never need more than 3 or 4 iterations.
            for j in range(0, len(remaining), 2):
                    if remaining[j] == remaining[j+1]:
                        print("Duplicate in e_i.e_j:", remaining, ", shuffling.")
                        valid = False
                        break
                    else:
                        valid = True
            if valid:
                break
                #print("No duplicates in e_i.e_j:", remaining)
            else:
                valid = True
                random.shuffle(remaining)
        
        i1 = remaining[2*i]
        i2 = remaining[2*i + 1]
        e1 = sp.Symbol(f"e{i1}", commutative=True)
        e2 = sp.Symbol(f"e{i2}", commutative=True)
        factors.append(DP(e1, e2))

    # Now build the factors of p_i.p_j to ensure the correct mass dimension
    # We just pick y random pairs of momentum indices in [1..n] and dot them together.
    for _ in range(y):
        i1 = random.randint(1, n)
        i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
        p1 = sp.Symbol(f"p{i1}", commutative=True)
        p2 = sp.Symbol(f"p{i2}", commutative=True)
        factors.append(DP(p1, p2))
    
    # Shuffle all factors for a bit of spice and multiply them into a single expression
    random.shuffle(factors)
    if not factors:
        return sp.Integer(1)
    
    monomial = sp.Mul(*factors)  # multiply all factors together
    
    return monomial

def generate_denominator(n, dim):
    """
    For now, we only consider momentum in the denominator (Mandelstams, essenially), so 
    we generate a product of d/2 momentum dot products.
    
    """
    denom = []
    for _ in range(dim//2): # We want im/2 pairs of momentum indices. dim should be even!
        i1 = random.randint(1, n)
        i2 = random.choice([mn for mn in range(1, n+1) if mn != i1])
        p1 = sp.Symbol(f"p{i1}", commutative=True)
        p2 = sp.Symbol(f"p{i2}", commutative=True)
        denom.append(DP(p1, p2))
    random.shuffle(denom)
    return sp.Mul(*denom)

def generate_amplitude(n, n_gluons, n_gravitons, dim, max_monomials=3):
    """
    Generate a random amplitude with n external particles (n_gluons gluons and n_gravitons gravitons), mass-dimension dim, with up to max_monomials monomials.
    
    We will choose the denominator have mass dimension 2(n-3) and therefore the numerator to have mass dimension dim + 2(n-3). 
    """
    i = random.choice(range(1, max_monomials+1))
    
    monos = []
    for _ in range(1, i+1):
        monos.append(generate_monomial(n, n_gluons, n_gravitons, dim + 2*(n-3)))
    #print("Monomials:", monos)
    numerator = sp.Add(*monos)
    denominator = generate_denominator(n, 2*(n-3))
    return numerator/denominator

In [11]:
def momentum_conservation_substitution(j, n):
    """
    Substitute p_j = - sum_{k != j}^n p_k.
    
    """
    p_syms = [sp.Symbol(f"p{i}", commutative=True) for i in range(1, n+1)] # sympy symbols for p1, p2, ..., p_n
    replacement = -sum(p_syms[k] for k in range(n) if (k != (j-1)))
    return {p_syms[j-1]: replacement}

expr = DP(sp.Symbol("p1"),sp.Symbol("p2")).subs(momentum_conservation_substitution(2, 5)) # Substitute p_2 = -p_1 - p_3 - p_4 - p_5
print(distribute_DP(expr))

-DP(p1, p1) - DP(p1, p3) - DP(p1, p4) - DP(p1, p5)


In [12]:

def single_scramble(expr, n, n_gluons, n_gravitons):
    """
    Perform ONE random scramble operation on 'expr':
      1) Multiply by 1 using (e_i.p_j)/(e_i.p_j), applying momentum
         conservation in the numerator only.
      2) Add zero using e_i.p_i=0 and momentum conservation on p_i in the numerator.
      3) Directly apply momentum conservation to some p_j in the entire expression.

    If n_gluons==0 and n_gravitons==0, we only do operation #3 (scalars).
    """
    # If no polarizations, skip 1 and 2:
    if n_gluons == 0 and n_gravitons == 0:
        # Operation 3 only
        operation = 3
    else:
        operation = random.choice([1, 2, 3])
    
    total_polarizations = n_gluons + n_gravitons
    
    if operation == 1:
        # --- Multiply by 1 using ( e_i . p_j ) / ( e_i . p_j ) ---
        i = random.randint(1, total_polarizations)
        j = random.randint(1, n)

        e_i = sp.Symbol(f"e{i}", commutative=True)
        p_j = sp.Symbol(f"p{j}", commutative=True)

        # Numerator with momentum conservation on p_j
        subs_dict = momentum_conservation_substitution(j, n)
        num_expr = distribute_DP(DP(e_i, p_j).subs(subs_dict))
        den_expr = DP(e_i, p_j)

        return expr * (num_expr/den_expr)

    elif operation == 2:
        # --- Add zero using e_i.p_i=0, then momentum conservation on p_i ---
        i = random.randint(1, total_polarizations)

        # We want a momentum index j != i only if i <= n. Otherwise any j in 1..n is fine.
        # But let's handle corner cases gracefully.
        if i <= n:
            possible_js = [x for x in range(1, n+1) if x != i]
            if not possible_js:
                possible_js = list(range(1, n+1))  # fallback
        else:
            possible_js = list(range(1, n+1))

        j = random.choice(possible_js)

        e_i = sp.Symbol(f"e{i}", commutative=True)
        p_i = sp.Symbol(f"p{i}", commutative=True)  # only valid if i <= n
        p_j = sp.Symbol(f"p{j}", commutative=True)

        # ( e_i . p_i ), with p_i replaced by sum_{k != i}, then / ( e_i . p_j )
        subs_dict = {}
        if i <= n:
            subs_dict = momentum_conservation_substitution(i, n)

        zero_num_subbed = distribute_DP(DP(e_i, p_i).subs(subs_dict))
        zero_term = zero_num_subbed / DP(e_i, p_j)

        # Combine over common denominator e_i . p_j
        common_denom = DP(e_i, p_j)
        expr_as_fraction = expr * (common_denom / common_denom)

        return expr_as_fraction + zero_term

    else:
        # --- Operation 3: Direct momentum conservation on some p_j in the entire expression ---
        j = random.randint(1, n)
        subs_dict = momentum_conservation_substitution(j, n)
        return distribute_DP(expr.subs(subs_dict))

def scramble(amp, n, n_gluons=0, n_gravitons=0,
             number_of_scrambles=1, only_scramble_numerator=True):
    """
    Repeatedly scramble the amplitude 'amp' using the three operations.
    
    Parameters:
      amp : A Sympy expression (the amplitude).
      n   : Number of external momenta p1, p2, ..., p_n.
      n_gluons, n_gravitons : # of gluons and gravitons, used for picking e_i indices.
                              If both are 0, only operation #3 is used (scalars).
      dim : (Optional) overall dimension, not strictly needed but included for consistency.
      only_scramble_numerator : If True, apply each scramble only to the numerator of 'amp'.
      number_of_scrambles     : How many times to apply a single scramble in succession.

    Returns:
      scrambled_amp: The final scrambled amplitude after 'number_of_scrambles' operations.
    """
    scrambled_amp = amp

    for _ in range(number_of_scrambles):
        if only_scramble_numerator:
            # Separate numerator and denominator
            num, den = sp.fraction(scrambled_amp)
            # Apply a single scramble to the numerator only
            new_num = single_scramble(num, n, n_gluons, n_gravitons)
            scrambled_amp = new_num / den
        else:
            # Scramble the entire expression
            scrambled_amp = single_scramble(scrambled_amp, n, n_gluons, n_gravitons)
    
    return scrambled_amp

In [13]:

amp = generate_amplitude(n=5,n_gluons=5,n_gravitons=0,dim=1,max_monomials=1)
print("Original amplitude:")
sp.pprint(amp)
scrambled_amp = scramble(amp, n=5, n_gluons=5, n_gravitons=0, number_of_scrambles=1)
print("Scrambled amplitude:")
sp.pprint(sp.expand(scrambled_amp))

# Can we tokenize this?
tokens = tokenize_expr(scrambled_amp)
print(tokens)
# And reconstruct it?
reconstructed = detokenize_expr(tokens)
print("Reconstructed amplitude:")
print(reconstructed - scrambled_amp) # Should be zero

Original amplitude:
e₁⋅e₃⋅e₂⋅p₃⋅e₅⋅e₄⋅p₁⋅p₃⋅p₂⋅p₃
─────────────────────────────
         p₂⋅p₅⋅p₃⋅p₅         
Scrambled amplitude:
e₁⋅e₃⋅e₂⋅p₃⋅e₅⋅e₄⋅p₁⋅p₃⋅p₂⋅p₃        1              e₄⋅p₂               e₄⋅p₃  ↪
───────────────────────────── - ─────────── - ───────────────── - ──────────── ↪
         p₂⋅p₅⋅p₃⋅p₅            p₂⋅p₅⋅p₃⋅p₅   e₄⋅p₁⋅p₂⋅p₅⋅p₃⋅p₅   e₄⋅p₁⋅p₂⋅p₅⋅ ↪

↪               e₄⋅p₅      
↪ ───── - ─────────────────
↪ p₃⋅p₅   e₄⋅p₁⋅p₂⋅p₅⋅p₃⋅p₅
['*', '(', '^', '(', 'DP', '(', 'p2', ',', 'p5', ')', ',', '-1', ')', ',', '^', '(', 'DP', '(', 'p3', ',', 'p5', ')', ',', '-1', ')', ',', '+', '(', '*', '(', '^', '(', 'DP', '(', 'e4', ',', 'p1', ')', ',', '-1', ')', ',', '+', '(', '*', '(', '-1', ',', 'DP', '(', 'e4', ',', 'p1', ')', ')', ',', '*', '(', '-1', ',', 'DP', '(', 'e4', ',', 'p2', ')', ')', ',', '*', '(', '-1', ',', 'DP', '(', 'e4', ',', 'p3', ')', ')', ',', '*', '(', '-1', ',', 'DP', '(', 'e4', ',', 'p5', ')', ')', ')', ')', ',', '*', '(', 'DP', '(', 'e1', ',', 'e3', ')',

In [14]:
# Let's generate some data and save it to disk
# First we need to build a vocabulary, as we want to convert the expressions to token indices.
def is_number_str(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

# Example fixed vocabulary with momentum and polarization tokens and some symbols.
vocab = {
    "PAD": 0,
    "UNK": 1,
    "SOS": 2,
    "EOS": 3,
}

# Momentum tokens: "p1" to "p10" mapped to indices 4 to 13.
for i in range(1, 11):
    vocab[f"p{i}"] = 3 + i  # 4, 5, ..., 13

# Polarization tokens: "e1" to "e11" mapped to indices 14 to 24.
for i in range(1, 12):
    vocab[f"e{i}"] = 13 + i  # 14, 15, ..., 24
    
# Add tokens for the function name and common symbols.
symbol_tokens = ["DP", "(", ")", ",", "+", "-", "*", "/", "^"]
start_index = len(vocab)
for token in symbol_tokens:
    vocab[token] = start_index
    start_index += 1

# Include a bunch of numbers as well, from -10 to 10. If we go outside this, then we'll have to use the UNK token. Probably bad.    
for i in range(-10, 11):
    vocab[str(i)] = start_index
    start_index += 1

import json
vocab_json = json.dumps(vocab)
print("Vocab JSON:")
print(vocab_json)
    
  

Vocab JSON:
{"PAD": 0, "UNK": 1, "SOS": 2, "EOS": 3, "p1": 4, "p2": 5, "p3": 6, "p4": 7, "p5": 8, "p6": 9, "p7": 10, "p8": 11, "p9": 12, "p10": 13, "e1": 14, "e2": 15, "e3": 16, "e4": 17, "e5": 18, "e6": 19, "e7": 20, "e8": 21, "e9": 22, "e10": 23, "e11": 24, "DP": 25, "(": 26, ")": 27, ",": 28, "+": 29, "-": 30, "*": 31, "/": 32, "^": 33, "-10": 34, "-9": 35, "-8": 36, "-7": 37, "-6": 38, "-5": 39, "-4": 40, "-3": 41, "-2": 42, "-1": 43, "0": 44, "1": 45, "2": 46, "3": 47, "4": 48, "5": 49, "6": 50, "7": 51, "8": 52, "9": 53, "10": 54}


In [32]:
  
def tokens_to_numpy_fixed(tokens, vocab=vocab):
    """
    Convert a tokenized sequence (list of tokens) to a NumPy array of type int32.
    For tokens representing numbers, if the literal is in the vocabulary it is used;
    otherwise, we map it to a special token, e.g., <NUM>, which we add to the vocabulary.
    """
    # Define the special numeric token, if not already in the vocabulary.
    special_num_token = "<NUM>"
    if special_num_token not in vocab:
        vocab[special_num_token] = len(vocab)
    
    token_indices = []
    for token in tokens:
        # If the token is a number and not in the vocabulary, map to <NUM>
        if is_number_str(token) and token not in vocab:
            token_indices.append(vocab[special_num_token])
        else:
            token_indices.append(vocab.get(token, vocab["UNK"]))
    return np.array(token_indices, dtype=np.int32)

def generate_dataset(num_samples, n, n_gluons=0, n_gravitons=0, dim=1, max_monomials=1, scrambles=3):
    """
    For each sample, generate a simple amplitude expression and a scrambled version,
    tokenize them, convert tokens to numpy int32 arrays using a fixed vocabulary, and return
    the resulting lists of integer sequences.
    
    Returns:
      (list of numpy arrays for simple expressions, list of numpy arrays for scrambled expressions)
    """
    simple_data = []
    scrambled_data = []
    for _ in range(num_samples):
        # Generate a simple amplitude expression (a Sympy object)
        simple_expr = generate_amplitude(n, n_gluons, n_gravitons, dim, max_monomials)
        # Tokenize the expression (your custom tokenizer)
        simple_tokens = tokenize_expr(simple_expr)
        # Convert token list to numpy int32 array using the fixed vocabulary.
        simple_ids = tokens_to_numpy_fixed(simple_tokens, vocab)
        
        # Now generate the scrambled version.
        scrambled_expr = scramble(simple_expr, n, n_gluons, n_gravitons, number_of_scrambles=scrambles)
        scrambled_tokens = tokenize_expr(scrambled_expr)
        scrambled_ids = tokens_to_numpy_fixed(scrambled_tokens, vocab)
        
        simple_data.append(simple_ids)
        scrambled_data.append(scrambled_ids)
    return simple_data, scrambled_data

def write_dataset_to_hdf5(filename, simple_data, scrambled_data):
    """
    Writes the dataset to an HDF5 file with two datasets: 'simple' and 'scrambled'.
    Each data item is a variable-length numpy array of int32 (i.e. token indices).
    """
    # Define a special dtype for variable-length int32 arrays.
    dt = h5py.special_dtype(vlen=np.dtype('int32'))
    
    # Convert lists to object arrays so that each element is an array.
    simple_data_arr = np.empty(len(simple_data), dtype=object)
    simple_data_arr[:] = simple_data
    scrambled_data_arr = np.empty(len(scrambled_data), dtype=object)
    scrambled_data_arr[:] = scrambled_data
    
    with h5py.File(filename, "w") as f:
        f.create_dataset("simple", data=simple_data_arr, dtype=dt)
        f.create_dataset("scrambled", data=scrambled_data_arr, dtype=dt)
        f.attrs["vocab"] = vocab_json # We save the vocabulary along with the file as an attribute. This means we don't need to lead it back into pytorch later...
    print("Dataset saved to", filename)


# Parameters for dataset generation.
num_samples = 1000    # Total number of amplitude examples.
n_min = 3              # Minimum number of external particles. 
n_max = 8
n_gluons_max = 8       # Maximum number of external gluons.
n_gluons_min = 1
n_gravitons_max = 0    # Maximum number of external gravitons.
n_gravitons_min = 0       
max_monomials = 3      # Number of terms summed in the simple amplitude.
scramble_times = 3     # Number of scrambling operations applied.
dim = 1                # Mass dimension of the amplitude.
# For example, here we loop over a range of particle numbers with random number of gluons and gravitons between min and max.
i = n_min
while i <= n_max:
    # Generate the dataset.
    # Pick a random number of gluons and gravitons between min and max.
    # Ensure that at least n_gravitons_min particles remain for gravitons.
    n_gluons = random.randint(n_gluons_min, min(n_gluons_max, i - n_gravitons_min))
    n_gravitons = random.randint(n_gravitons_min, min(n_gravitons_max, i - n_gluons))
    dim = 4-i # Mass dimension is 4 - n_external_particles
    simple_data, scrambled_data = generate_dataset(
        num_samples,
        n=i,
        n_gluons=n_gluons,
        n_gravitons=n_gravitons,  # Since max gravitons is 0 in this example.
        dim=dim,
        max_monomials=max_monomials,
        scrambles=scramble_times
    )
    # Write the dataset to an HDF5 file.
    output_filename = f"amplitude_{i}_particle_dataset.hdf5"
    write_dataset_to_hdf5(output_filename, simple_data, scrambled_data)
    i += 1


Dataset saved to amplitude_3_particle_dataset.hdf5
Dataset saved to amplitude_4_particle_dataset.hdf5
Dataset saved to amplitude_5_particle_dataset.hdf5
Dataset saved to amplitude_6_particle_dataset.hdf5
Dataset saved to amplitude_7_particle_dataset.hdf5
Dataset saved to amplitude_8_particle_dataset.hdf5


In [31]:
amp = generate_amplitude(n=5,n_gluons=5,n_gravitons=0,dim=1,max_monomials=1)
print("Original amplitude:")
print(amp)
print("Scrambled amplitude:")
print(scramble(amp, n=5, n_gluons=5, n_gravitons=0, number_of_scrambles=100))

Original amplitude:
DP(e1, p2)*DP(e2, p1)*DP(e3, p2)*DP(e5, e4)*DP(p4, p2)/(DP(p1, p2)*DP(p4, p5))
Scrambled amplitude:
((((((((((-DP(e4, p1) - DP(e4, p2) - DP(e4, p3) - DP(e4, p5))/DP(e4, p5) + (-DP(e4, p1) - DP(e4, p2) - DP(e4, p3) - DP(e4, p5))/DP(e4, p2) + ((-DP(p1, p2) - DP(p2, p2) - DP(p3, p2) - DP(p5, p2))*DP(e1, p2)*DP(e2, p1)*DP(e3, p2)*DP(e5, e4) + DP(e2, p2)/DP(e2, p5) + DP(e3, p3)/DP(e3, p5) + 3*DP(e5, p5)/DP(e5, p2) + DP(e3, p3)/DP(e3, p2) + DP(e5, p5)/(-DP(e5, p1) - DP(e5, p2) - DP(e5, p3) - DP(e5, p5)))*DP(e3, p3) + DP(e3, p3)/(-DP(e3, p1) - DP(e3, p2) - DP(e3, p3) - DP(e3, p5)) + DP(e1, p1)/(-DP(e1, p1) - DP(e1, p2) - DP(e1, p3) - DP(e1, p5)))*DP(e4, p2) + DP(e5, p5)/(-DP(e5, p1) - DP(e5, p2) - DP(e5, p3) - DP(e5, p5)) + DP(e3, p3)/(-DP(e3, p1) - DP(e3, p2) - DP(e3, p3) - DP(e3, p5)))*(-DP(e3, p1) - DP(e3, p2) - DP(e3, p3) - DP(e3, p5))*DP(e2, p5) + 2*DP(e1, p1)/DP(e1, p5) + DP(e2, p2)/(-DP(e2, p1) - DP(e2, p2) - DP(e2, p3) - DP(e2, p5)))*DP(e2, p2)*DP(e2, p3)*DP(e3, p5